# 🔧 Fix YOLO-ESI ONNX Export — Kaggle Dataset

This notebook:
1. Clones the sonarvision repo
2. Uploads the merged Kaggle dataset (1 class: marine_debris)
3. Fixes the `patch_trainer` bug (nc mismatch)
4. Trains YOLOv8-ESI on Kaggle data
5. Exports a **new** ONNX with correct nc=1

**Local ONNX files are NOT touched.**

---

### Upload before running:
- `kaggle_merged.zip` — from `datasets/kaggle_merged/`

In [ ]:
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Upload kaggle_merged.zip from your local datasets/kaggle_merged/ directory
from google.colab import files
print('Upload kaggle_merged.zip:')
uploaded = files.upload()

!unzip -q kaggle_merged.zip -d /content/
!sed -i 's|path:.*|path: /content/kaggle_merged|' /content/kaggle_merged/data.yaml

import yaml
from pathlib import Path
with open('/content/kaggle_merged/data.yaml') as f:
    cfg = yaml.safe_load(f)
print(f'Dataset: {cfg["nc"]} class, names={cfg["names"]}')
print(f'Train: {len(list(Path(cfg["path"], "train", "images").glob("*")))} images')
print(f'Val: {len(list(Path(cfg["path"], "valid", "images").glob("*")))} images')

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.block import C2f

# ── SE Block (Squeeze-and-Excitation) ──
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(c, c//r, bias=False), nn.ReLU(),
            nn.Linear(c//r, c, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        w = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * w

# ── C2f with SE attention ──
class C2fSE(nn.Module):
    def __init__(self, c2f, se):
        super().__init__()
        self.c2f = c2f
        self.se = se
        self.i = c2f.i
        self.f = c2f.f
    def forward(self, x):
        return self.se(self.c2f(x))

# ── Build YOLOv8-ESI ──
def build_esi():
    yolo = YOLO('yolov8n.pt')
    layers = list(yolo.model.model)
    for i, layer in enumerate(layers):
        if isinstance(layer, C2f):
            c2 = layer.cv2.conv.out_channels
            se = SEBlock(c2)
            se.i, se.f, se.type = i, getattr(layer, 'f', -1), 'SEBlock'
            layers[i] = C2fSE(layer, se)
    yolo.model.model = nn.Sequential(*layers)
    total = sum(p.numel() for p in yolo.model.parameters())
    print(f'YOLOv8-ESI built: {total/1e6:.2f}M params')
    return yolo.model

print('✓ Custom modules loaded')

In [ ]:
from ultralytics.models.yolo.detect.train import DetectionTrainer

DATA = '/content/kaggle_merged/data.yaml'

def patch_trainer_fixed(model_obj):
    """FIXED patch: sets nc=1 BEFORE creating the DetectionModel."""
    _orig = DetectionTrainer.get_model
    def _patched(self, cfg=None, weights=None, verbose=True):
        from ultralytics.nn.tasks import DetectionModel
        from ultralytics.utils import RANK

        # Create DetectionModel with nc=1 — this creates a 1-class detection head
        dm = DetectionModel(cfg, nc=1, ch=self.data.get('channels', 3),
                            verbose=verbose and RANK == -1)

        # Copy backbone + neck from ESI (layers 0-8), skip detection head (layers 9+)
        esi_layers = list(model_obj.model)
        dm_layers = list(dm.model)
        for i in range(min(len(esi_layers), len(dm_layers))):
            if i < 9:  # Keep ESI backbone/neck
                dm_layers[i] = esi_layers[i]
            # Layers 9+ (detection head) are kept from dm — nc=1 ✓

        dm.model = nn.Sequential(*dm_layers)
        dm.nc = 1
        dm.names = {0: 'marine_debris'}

        return dm
    DetectionTrainer.get_model = _patched
    return _orig

print('✓ Fixed patch_trainer loaded')

In [ ]:
print('='*60)
print('TRAINING: YOLOv8-ESI on Kaggle (1 class: marine_debris)')
print('='*60)

esi_obj = build_esi()
_orig = patch_trainer_fixed(esi_obj)

try:
    model = YOLO('yolov8n.pt')
    model.train(
        data=DATA,
        epochs=50,
        imgsz=256,
        batch=32,
        patience=15,
        lr0=0.01,
        lrf=0.01,
        warmup_epochs=2,
        freeze=10,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='esi_kaggle_fixed',
        project='/content/runs',
        exist_ok=True,
        plots=True,
    )
finally:
    DetectionTrainer.get_model = _orig

print('\n✓ Training complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# VALIDATE + EXPORT
# ═══════════════════════════════════════════════════════════

import shutil
from pathlib import Path

BEST = '/content/runs/esi_kaggle_fixed/weights/best.pt'

# Validate
model = YOLO(BEST)
best_f1, best_conf = 0, 0.05
for conf in [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
    r = model.val(data=DATA, imgsz=256, conf=conf, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    if f1 > best_f1:
        best_f1, best_conf = f1, conf

r = model.val(data=DATA, imgsz=256, conf=best_conf, verbose=False)
p, rv = r.box.mp, r.box.mr
f1 = 2*p*rv / max(p+rv, 1e-8)
print(f'Best conf: {best_conf}')
print(f'mAP50: {r.box.map50:.4f}')
print(f'Precision: {p:.4f}')
print(f'Recall: {rv:.4f}')
print(f'F1: {f1:.4f}')

# Verify output shape before export
print(f'\nModel nc: {model.model.nc}')
print(f'Model names: {model.model.names}')

# Export FP16 ONNX
export_path = model.export(format='onnx', imgsz=256, half=True, simplify=True)
print(f'\nExported: {export_path}')

# Verify ONNX output shape
import onnxruntime as ort
session = ort.InferenceSession(export_path, providers=['CPUExecutionProvider'])
out_shape = session.get_outputs()[0].shape
nc_out = out_shape[1] - 4
print(f'ONNX output: {out_shape} -> nc={nc_out}')
assert nc_out == 1, f'FAIL: Expected nc=1, got nc={nc_out}'
print('✓ ONNX has correct nc=1 (marine_debris)')

In [ ]:
# Download the fixed ONNX
from google.colab import files

EXPORT_DIR = Path('/content/runs/esi_kaggle_fixed/weights/')
onnx_files = list(EXPORT_DIR.glob('*.onnx'))

# Also copy to a clean name
shutil.copy2(export_path, '/content/yolo_esi_kaggle_fixed.onnx')

print('Download the fixed ONNX:')
files.download('/content/yolo_esi_kaggle_fixed.onnx')

# Also download the .pt for reference
files.download(BEST)